In [9]:
# Import Statement
import os
import h5py
import numpy as np
import pandas as pd
import re

In [11]:
# CHANGE HERE

# Read in labeled folder names and timestamps
Cluster_detail_results = pd.read_csv( os.path.join(r'Y:\3darena_behavior\mitopark\mitopark_042025\042025_2mp\week8-10-12-14-23Comp_video\Combined_Results\Results\test1\Cluster_detail_results.csv') )

# Read in mat file
filename = r"Y:\3darena_behavior\mitopark\mitopark_042025\042025_2mp\week8-10-12-14-23Comp_video\Combined_Results\Results\test1\session_1_out.mat"

# Output file name
output_path = r"D:\AccelClusterVisualizations\data"

# Use any label for mouse
mouse_name = "2mp"  

In [12]:
# Read in values from clustersIdx, set automatically to the cluster value it represents rather than the actual value of the cell
if os.path.exists(filename):
    with h5py.File(filename, 'r') as file:
        library = file['Library']
        clustersIdx = library['clustersIdx']
        
        group_labels = []
        for i in range(clustersIdx.shape[0]):
            idxref = clustersIdx[i, 0]
            cluster_indices = np.array(file[idxref]).flatten()
            group_labels.extend([i+1] * len(cluster_indices))
        
        group_labels_df = pd.DataFrame(group_labels, columns=['GroupLabel'])
        print(f"Group labels shape: {group_labels_df.shape}")
else:
    raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')

Group labels shape: (121067, 1)


In [13]:
# Read in the feature values from clusters and stack on top of each other
if os.path.exists(filename):
    print('Loading session file from expected Results/test1 directory')
    with h5py.File(filename, 'r') as file:
        if 'Library' not in file:
            raise KeyError("The key 'Library' was not found in the session file.")
        library = file['Library']
        clusters = library['clusters']
        
        features_data = []
        
        for i in range(clusters.shape[0]):
            cref = clusters[i, 0]
            feature_data = np.array(file[cref]).T 
            features_data.append(feature_data)
        
        features_matrix = np.vstack(features_data)
        features_df = pd.DataFrame(features_matrix)
        print(f"Loaded {len(features_data)} clusters with total points: {features_matrix.shape[0]}")
        print(f"Combined matrix shape: {features_matrix.shape}")
else:
    raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')
features_df['cluster'] = group_labels_df['GroupLabel'].values

Loading session file from expected Results/test1 directory
Loaded 97 clusters with total points: 121067
Combined matrix shape: (121067, 30)


In [14]:
# Create combined matrix by matching first occurrence of each cluster in features_df to first occurrence in Cluster_detail_results, keeping order of Cluster_detail_results
def match_features_to_details(features_df, detail_df):
    matched_rows = []
    cluster_counts = {}
    for idx, detail_row in detail_df.iterrows():
        cluster_val = detail_row['ClusterIdx']
        count = cluster_counts.get(cluster_val, 0)
        feature_rows = features_df[features_df['cluster'] == cluster_val]
        if count < len(feature_rows):
            feature_row = feature_rows.iloc[count]
            combined_row = feature_row.to_dict()
            combined_row['Timestamp'] = detail_row['Timestamp'] if 'Timestamp' in detail_row else None
            combined_row['ClusterIdx'] = cluster_val
            combined_row['Folder_Name'] = detail_row['Folder_Name'] if 'Folder_Name' in detail_row else None
            matched_rows.append(combined_row)
            cluster_counts[cluster_val] = count + 1
        else:
            combined_row = {col: None for col in features_df.columns}
            combined_row['Timestamp'] = detail_row['Timestamp'] if 'Timestamp' in detail_row else None
            combined_row['ClusterIdx'] = cluster_val
            combined_row['Folder_Name'] = detail_row['Folder_Name'] if 'Folder_Name' in detail_row else None
            matched_rows.append(combined_row)
    return pd.DataFrame(matched_rows)

combined_matrix = match_features_to_details(features_df, Cluster_detail_results)

In [ ]:
# Automatically detects Week_Number from Folder_Name based on pattern of 'weekXX'

def get_week_number(folder_name):
    if isinstance(folder_name, str):
        match = re.search(r'week(\d+)', folder_name)
        if match:
            return int(match.group(1))
    return None

combined_matrix['Week_Number'] = combined_matrix['Folder_Name'].apply(get_week_number)
combined_matrix['Week_Number'] = combined_matrix['Week_Number'].fillna(0).astype(int)
combined_matrix = combined_matrix[combined_matrix['Week_Number'] != 0]

In [18]:
# Clean up the finalized combined df (Column Titles : "Feature 1" ... "Feature 30" , "Timestamp" , "Cluster", "Week_Number")
feature_cols = {i: f'Feature{i+1}' for i in range(30)}
combined_matrix = combined_matrix.rename(columns=feature_cols)
combined_matrix = combined_matrix.drop(columns=['cluster'])
combined_matrix = combined_matrix.drop(columns=['Folder_Name'])
combined_matrix = combined_matrix.rename(columns={'ClusterIdx': 'Cluster'})

In [8]:
# Save combined_matrix file
output_file = os.path.join(output_path, f"combined_matrix_{mouse_name}.csv")
combined_matrix.to_csv(output_file, index=False)
print(f"Saved as {output_file}")
print(combined_matrix.head())

Saved as D:\AccelClusterVisualizations\data\combined_matrix_2mp.csv
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  \
0       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
1       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
2       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
3       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
4       0.0       0.0       0.0       0.0       0.0       0.0       0.0   

   Feature8  Feature9  Feature10  ...  Feature24  Feature25  Feature26  \
0  0.016667  0.983333   0.000000  ...   0.150000   0.366667   0.366667   
1  0.000000  0.033333   0.966667  ...   0.066667   0.316667   0.300000   
2  0.000000  0.000000   1.000000  ...   0.233333   0.350000   0.283333   
3  0.000000  0.000000   1.000000  ...   0.100000   0.400000   0.350000   
4  0.000000  0.000000   1.000000  ...   0.133333   0.266667   0.416667   

   Feature27  Feature28  Feature29  